# 问题1：第一阶段B续跑——保留未知时刻，完成未执行查询

使用 Python for Data Analysis (pda312)，从头依次运行；不安装新依赖。
复用原运行 user_stage1b_20261105_231816_900405 的 X11—X14 原始响应，不重新请求这4组列车数据。
仍会获取当前查询页面、站点字典及字段脚本，随后只新增广州南→珠海、珠海→横琴、横琴→珠海3组列车查询。
运行结果另存时间戳目录，原Notebook、原始响应和22条有效记录均保留。

已定位的两条记录：11月14日 Z8981、Y839，网站返回“列车运行图调整，暂停发售”以及24:00/24:00/99:59。
修复将该明确组合标为时刻未知，保存到 unavailable_timetables.csv，不将其换成真实时间。
若未知时刻仍存在，完整性标记仍为False，最短值仍为空；这是正确的数据边界，不是又一次程序失败。


In [1]:
# 问题1—辅助代码1：检查内核并加载本批代码
import sys, json, importlib
from pathlib import Path
import pandas as pd
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
assert Path(sys.executable).resolve() == Path(r"D:\anaconda\envs\pda312\python.exe").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
import stage1_collect
stage1_collect = importlib.reload(stage1_collect)
import stage1b_transfer
stage1b_transfer = importlib.reload(stage1b_transfer)
print("本批复用X11—X14，新增3组列车查询；未知时刻单独保存。")


本批复用X11—X14，新增3组列车查询；未知时刻单独保存。


## 问题1—核心代码4（续跑版）：重解析缓存，补做3组查询

输入：原A运行与原B运行的真实响应，以及原研究协议。输出：新时间戳目录下的CSV、原始新响应与复用来源清单。
应截图：代码、查询状态表、未公布时刻表与汇总；表格较多可分截图。
未知时刻记录保留车次、日期、原始占位值和来源；错误记录仍会触发停止。不要为取得完整标记而修改原始数据。


In [2]:
# 问题1—核心代码4（续跑版）：复用4组响应，仅补3组列车查询
summary_b = stage1b_transfer.collect_stage1b(
    actor="user", resume_run="user_stage1b_20261105_231816_900405")
OUT_B = Path(summary_b["output_dir"])
quality_b = pd.read_csv(OUT_B / "new_query_quality.csv", encoding="utf-8-sig")
display(quality_b[["query_id", "date", "status", "valid_count", "unavailable_count"]])
missing_b = pd.read_csv(OUT_B / "unavailable_timetables.csv", encoding="utf-8-sig")
display(missing_b[["query_date", "train_code", "source_message", "raw_departure", "raw_duration"]])
print("本批有效记录：", summary_b["new_train_records"])
print("缓存复用/新列车请求：", summary_b["reused_train_queries"], summary_b["network_train_queries"])
print("中转日期请求均成功返回：", summary_b["all_required_transfer_requests_returned"])
print("中转记录时刻均已知：", summary_b["all_required_transfer_timetables_available"])
print("程序错误：", summary_b["fatal_error"])
print("输出目录：", OUT_B)


X11 2026-11-11 西宁 → 拉萨: validated_records, valid=6, unavailable=0 (saved response)
X12 2026-11-12 西宁 → 拉萨: validated_records, valid=6, unavailable=0 (saved response)
X13 2026-11-13 西宁 → 拉萨: validated_records, valid=6, unavailable=0 (saved response)
X14 2026-11-14 西宁 → 拉萨: partial_timetable_unavailable, valid=4, unavailable=2 (saved response)
A01 2026-11-10 广州南 → 珠海: validated_records, valid=135, unavailable=0 (new query)
A02 2026-11-10 珠海 → 横琴: validated_records, valid=101, unavailable=0 (new query)
A03 2026-11-10 横琴 → 珠海: validated_records, valid=99, unavailable=0 (new query)


,query_id,date,status,valid_count,unavailable_count
0,X11,2026-11-11,validated_records,6,0
1,X12,2026-11-12,validated_records,6,0
2,X13,2026-11-13,validated_records,6,0
3,X14,2026-11-14,partial_timetable_unavailable,4,2
4,A01,2026-11-10,validated_records,135,0
5,A02,2026-11-10,validated_records,101,0
6,A03,2026-11-10,validated_records,99,0


,query_date,train_code,source_message,raw_departure,raw_duration
0,2026-11-14,Z8981,"列车运行图调整,暂停发售",24:00,99:59
1,2026-11-14,Y839,"列车运行图调整,暂停发售",24:00,99:59


本批有效记录： 357
缓存复用/新列车请求： 4 3
中转日期请求均成功返回： True
中转记录时刻均已知： False
程序错误： None
输出目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1b_20261105_234442_790924


## 问题1—核心代码5：保留已知候选，禁止冒称最短

下方使用原来的同站配对和门槛对比代码。未知时刻行不参与计算，也不以0、均值或惩罚值补齐。
缺失尚未解决时，candidate_count只是已知记录形成的候选数，total_min等字段保持空值。
请保留这一真实输出，不必反复查询直到字段有数值。


In [3]:
# 问题1—核心代码5：检查等待时间只计入一次，再比较门槛
pairs_b = pd.read_csv(OUT_B / "all_same_station_pairs.csv", encoding="utf-8-sig")
scenarios_b = pd.read_csv(OUT_B / "transfer_scenarios.csv", encoding="utf-8-sig")
ordered_b = pairs_b.loc[pairs_b["time_order_ok"].astype(str).str.lower().eq("true")]
assert (ordered_b["total_min"] == ordered_b["first_duration_min"] +
        ordered_b["wait_min"] + ordered_b["second_duration_min"]).all()
display(scenarios_b[["transfer_threshold_min", "candidate_count", "status",
                     "total_min", "wait_min", "first_train", "second_train"]])
print("30/60/90分钟是筛选门槛；等待已计入总历时，不能再额外加一次门槛。")
print("官方换乘可行性已核验：", summary_b["official_transfer_feasibility_verified"])


,transfer_threshold_min,candidate_count,status,total_min,wait_min,first_train,second_train
0,30,102,incomplete_query_coverage_no_best_claim,NaN,NaN,NaN,NaN
1,60,99,incomplete_query_coverage_no_best_claim,NaN,NaN,NaN,NaN
2,90,98,incomplete_query_coverage_no_best_claim,NaN,NaN,NaN,NaN


30/60/90分钟是筛选门槛；等待已计入总历时，不能再额外加一次门槛。
官方换乘可行性已核验： False


## 问题1—核心代码6：保持完整性门槛

保留原代码，仅当日期和时刻覆盖均满足当前检查时才输出代表结果。
当前2条时刻未知，因此预期显示“不报告最短值”的提示。3组接入查询可继续完成，与此门槛独立。


In [4]:
# 问题1—核心代码6：展示60分钟换乘假设下的候选
eligible_b = pairs_b.loc[(pairs_b["classification"] == "time_ordered_transfer_candidate") &
                         (pairs_b["wait_min"] >= 60)]
if summary_b["all_required_transfer_dates_queried_successfully"] and not eligible_b.empty:
    best_b = eligible_b.sort_values(["total_min", "departure_at", "first_train", "second_train"]).head(1)
    display(best_b[["first_train", "second_train", "from_station", "transfer_station", "to_station"]])
    display(best_b[["departure_at", "transfer_arrival_at", "transfer_departure_at", "arrival_at"]].T)
    display(best_b[["first_duration_min", "wait_min", "second_duration_min", "total_min"]])
    best_b.to_csv(OUT_B / "representative_transfer_60min.csv", index=False, encoding="utf-8-sig")
else:
    print("查询不完整或没有满足假设的候选；此时不能报告最短值或认定不可达。")


查询不完整或没有满足假设的候选；此时不能报告最短值或认定不可达。


## 回传材料

保存本Notebook，回传续跑版核心代码4的查询质量、未知时刻表和汇总截图；核心代码5—6的实际输出也请保留。
CSV在共享目录，可直接由助手核验。若新查询报错，请原样回传，不删除缺失记录或手工改日期。
修复说明位于作业目录 docs/stage1b_placeholder_fix.md。
